# CS-RHF Result comparison 
Results of the implementation compared to those obtained with [Qchem](https://doi.org/10.1080/00268976.2014.952696). 

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
from pyscf import gto
from py_mods.src.SCF.basis_utils import even_temp_uncontr_str
from py_mods.src.SCF.CSRHF import RHF_theta_traj, CS_RHF_ContextClass

# Definition of basis and intgrals 

In [ ]:
He_tempered_str = even_temp_uncontr_str('He', 'S', 7.668876968794860E-002, 1.9581497063588078, 21) # because this is the reference data 

mol_He_even= gto.M(atom = 'He 0 0 0', spin=0, charge=0,) # basis='aug-cc-pVqZ')

mol_He_even.basis = {'He': gto.basis.parse(He_tempered_str)}
mol_He_even.build()

T_even_H2 = mol_He_even.intor('int1e_kin')
V_even_H2 = mol_He_even.intor('int1e_nuc')
S_even_H2 = mol_He_even.intor('int1e_ovlp')
eri_even_H2 = mol_He_even.intor('int2e')

In [ ]:
large_basis = '''
He    S
      5.285000E+02           0.000000E+00           9.400000E-04           0.000000E+00           0.000000E+00
      7.931000E+01           0.000000E+00           7.214000E-03           0.000000E+00           0.000000E+00
      1.805000E+01           0.000000E+00           3.597500E-02           0.000000E+00           0.000000E+00
      5.085000E+00           0.000000E+00           1.277820E-01           0.000000E+00           0.000000E+00
      1.609000E+00           1.000000E+00           3.084700E-01           0.000000E+00           0.000000E+00
      5.363000E-01           0.000000E+00           4.530520E-01           1.000000E+00           0.000000E+00
      1.833000E-01           0.000000E+00           2.388840E-01           0.000000E+00           1.000000E+00
He    S
      0.0481900              1.0000000
He    P
      5.994000E+00           1.000000E+00           0.000000E+00           0.000000E+00
      1.745000E+00           0.000000E+00           1.000000E+00           0.000000E+00
      5.600000E-01           0.000000E+00           0.000000E+00           1.000000E+00
He    P
      0.1626000              1.0000000
He    D
      4.299000E+00           1.000000E+00           0.000000E+00
      1.223000E+00           0.000000E+00           1.000000E+00
He    D
      0.3510000              1.0000000
He    F
      2.680000E+00           1.0000000
He    F
      0.6906000              1.0000000
END
'''

mol_He_qz= gto.M(atom = 'He 0 0 0', spin=0, charge=0,) # basis='aug-cc-pVqZ')

mol_He_qz.basis = {'He': gto.basis.parse(large_basis)}
mol_He_qz.build()

T_qz_H2 = mol_He_qz.intor('int1e_kin')
V_qz_H2 = mol_He_qz.intor('int1e_nuc')
S_qz_H2 = mol_He_qz.intor('int1e_ovlp')
eri_qz_H2 = mol_He_qz.intor('int2e')


In [ ]:
def load_traj(filename):
    with open(filename) as f:
        cont = f.readlines()
    cont = [line.strip().replace('(', '').replace(')','') for line in cont]
    thetas = np.array([int(line.split(';')[0]) for line in cont])
    eners = [line.split(';')[1].strip().replace(' ', '').replace(',','+').replace('+-', '-') +'j' for line in cont]
    eners = np.array([complex(a) for a in eners])

    return thetas, eners

# 2s1 Helium 
## Even-tempered basis

In [ ]:
w1, k1 = load_traj('data/CSHF_QCHEM/He_1s2_eventemp_qchem.dat')

In [ ]:
He_2s2_even_gs_xt = CS_RHF_ContextClass(
    S_even_H2, 
    T_even_H2, 
    V_even_H2, 
    eri_even_H2, 
    2, 
    max_iter=1000, 
    threshold=2E-10, 
    conv_type='CROP'
)

traj_ener_1 = RHF_theta_traj(0.08, 9, He_2s2_even_gs_xt)

He_2s2_even_gs = np.array(traj_ener_1[1])

In [ ]:
plt.scatter(k1.real, k1.imag, label='Qchem data', marker='o', c='orange')
plt.scatter(He_2s2_even_gs.real, He_2s2_even_gs.imag,label='Implemetation',  c='RebeccaPurple', marker='x')
plt.legend()
plt.xlabel('Re(Ener)')
plt.ylabel('Im(Ener)')
plt.show()

In [ ]:
error = He_2s2_even_gs-k1
print(f'Mean error: {np.mean(error): 6.4E}')
print(f'Max error:  {np.max(error): 6.4E}')

## Augmented aug-cc-pV5Z

In [ ]:
w2, k2 = load_traj('data/CSHF_QCHEM/He_1s2_augqz_qchem.dat')

In [ ]:
He_2s2_qz_gs_ctx = CS_RHF_ContextClass(
    S_qz_H2, 
    T_qz_H2, 
    V_qz_H2, 
    eri_qz_H2, 
    n_electrons=2
)

traj_ener_2 = RHF_theta_traj(0.08, 9, He_2s2_qz_gs_ctx)

He_2s2_qz_gs = np.array(traj_ener_2[1])

In [ ]:
plt.scatter(k2.real, k2.imag, label='Qchem data', marker='o', c='orange')
plt.scatter(He_2s2_qz_gs.real, He_2s2_qz_gs.imag,label='Implemetation',  c='RebeccaPurple', marker='x')
plt.legend()
plt.xlabel('Re(Ener)')
plt.ylabel('Im(Ener)')
plt.show()

In [ ]:
error = He_2s2_qz_gs-k2
print(f'Mean error: {np.mean(error): 6.4E}')
print(f'Max error:  {np.max(error): 6.4E}')

# 2s2
## Even-tempered

In [ ]:
w3, k3 = load_traj('data/CSHF_QCHEM/He_2s2_eventemp_qchem.dat')

In [ ]:
He_2s2_even_cxt = CS_RHF_ContextClass(
    S_even_H2, 
    T_even_H2, 
    V_even_H2, 
    eri_even_H2, 
    2, 
    max_iter=1000, 
    threshold=2E-10, 
    conv_type='CROP'
)

He_2s2_even_cxt.occupation = np.array([0,2,0])
traj_ener_3 = RHF_theta_traj(0.4, 41, He_2s2_even_cxt)

He_2s2_even_ex = np.array(traj_ener_3[1])

In [ ]:
plt.scatter(k3.real, k3.imag, label='Qchem data', marker='o', c='orange')
plt.scatter(He_2s2_even_ex.real, He_2s2_even_ex.imag,label='Implemetation',  c='RebeccaPurple', marker='x')
plt.legend()
plt.xlabel('Re(Ener)')
plt.ylabel('Im(Ener)')
plt.show()

In [ ]:
error = He_2s2_even_ex-k3
print(f'Mean error: {np.mean(error): 6.4E}')
print(f'Max error:  {np.max(error): 6.4E}')

## 2s2 aug-cc-pV5Z

In [ ]:
w4, k4 = load_traj('data/CSHF_QCHEM/He_2s2_augqz_qchem.dat')

In [ ]:
He_2s2_qz_ctx = CS_RHF_ContextClass(
    S_qz_H2, 
    T_qz_H2, 
    V_qz_H2, 
    eri_qz_H2, 
    n_electrons=2
)

He_2s2_qz_ctx.occupation = np.array([0,2,0])

traj_ener_4 = RHF_theta_traj(0.08, 9, He_2s2_qz_ctx)

He_2s2_qz_ex = np.array(traj_ener_4[1])

In [ ]:
plt.scatter(k4.real, k4.imag, label='Qchem data', marker='o', c='orange')
plt.scatter(He_2s2_qz_ex.real, He_2s2_qz_ex.imag,label='Implemetation',  c='RebeccaPurple', marker='x')
plt.legend()
plt.xlabel('Re(Ener)')
plt.ylabel('Im(Ener)')
#plt.xlim([-0.7191, -0.7190])
plt.show()

In [ ]:
error = He_2s2_qz_ex-k4
print(f'Mean error: {np.mean(error): 6.4E}')
print(f'Max error:  {np.max(error): 6.4E}')